  <img src="../images/mackenzie_logo.jpg" width="25%" align="right"/>

# **PROJETO APLICADO IV - Ciência de Dados EaD - 2026/02**


# **Documentação do Projeto**

---

**Versão:** Etapa 2, entrega em 28/09/2026. O texto da Etapa 1 permanece. O incremento está no referencial, no diagrama, no cronograma e nas referências novas. A revisão da Etapa 2 também levou a correções no pipeline de origem e a uma nova exportação da base, com corte em 21/09/2026 (ver Descrição da base de dados).


# **Paridade de preços de combustíveis, atividade econômica e a ação da Petrobras**
---


In [ ]:
#@title **Identificação do Grupo**

#@markdown Integrantes do Grupo, nome completo em ordem alfabética (*informe: \<nome\>, \<matrícula\>*)
#@markdown AVA: grupo 19
Aluno1 = 'Guilherme Augusto Leal Oliveira, 10226108' #@param {type:"string"}
Aluno2 = 'Guilherme Rocha de Souza Duarte, 10730632' #@param {type:"string"}
Aluno3 = 'Ricardo Zulian de Souza Amaral, 10433651' #@param {type:"string"}
Aluno4 = '' #@param {type:"string"}
Aluno5 = '' #@param {type:"string"}


# **Resumo**

O preço doméstico de diesel e gasolina no Brasil se afasta com frequência da paridade internacional. Esse desvio, medido pelo PPI da ABICOM, altera custo de frete e mobilidade no Brasil como também o resultado financeiro e a cotação das ações da Petrobras. O objetivo do grupo é construir um produto de série temporal que estima a defasagem entre a paridade de preços e quatro alvos: o fechamento de PETR4, o preço de combustível da ANP e da Petrobras, os indicadores de tráfego, atividade e emprego, e o resultado trimestral da Petrobras. A base é um snapshot congelado de séries oficiais (ABICOM, ANP, Petrobras, B3, ABCR, IBGE e BCB) em SQLite, com âncora diária no PPI de 5 de janeiro de 2022 a 21 de setembro de 2026. A metodologia prevista alinhará frequências mistas e aplicará modelos estatísticos e de aprendizado de máquina (SARIMAX, LSTM e TimesFM) com variáveis exógenas. O produto apoia a leitura da política de preços e dos efeitos sobre trabalho, cidades e instituições (ODS 8, 11 e 16). Os resultados numéricos entram nas etapas 3 e 4.


# **Introdução**

## Contexto

O Brasil combina preço na refinaria com a paridade internacional como referência de mercado. O PPI (Índice de Paridade de Preços) da ABICOM publica, em dias úteis, o diferencial em R\$/L e em % entre o preço Petrobras e a paridade de importação para gasolina e diesel (ABICOM, 2026). Diesel move carga. Gasolina move o tráfego leve nas cidades. A Petrobras é empresa de economia mista listada na B3; o papel PETR4 reage a reajuste e à paridade. O tema é série temporal com frequências mistas: dia, semana, mês e trimestre.

O recorte liga três Objetivos de Desenvolvimento Sustentável. O ODS 8 trata de trabalho e crescimento: diesel, tráfego pesado, indústria e desemprego. O ODS 11 trata de cidades: gasolina, tráfego leve e preço ao consumidor. O ODS 16 trata de instituições: a política de preços de uma empresa estatal de capital aberto e misto e o PPI como sinal público do desvio em relação à paridade e ao interesse do acionista.

## Motivação

A política de preços da Petrobras mudou mais de uma vez nesta década. O mercado observa o preço internacional dos derivados de petróleo e o PPI, mas a transmissão até o posto, o volume e as variáveis sócio-economicas não são imediatas nem uniformes. Existe uma lacuna entre o preço do petróleo e seus derivados, o cartão diário de paridade e uma medida reproduzível de defasagem até atividade, desemprego e tráfego. São vária fontes públicas  (ANP, IBGE, ABCR, B3, BCB), porém em calendários distintos. O grupo junta essas séries em um recorte único a partir de 2022 e testa se o comportamento do petróleo e o PPI antecipam preço, volume, tráfego e o retorno de PETR4.

## Objetivo geral

O objetivo do projeto é medir a defasagem com que um choque de paridade e de preço de combustível, medido pelo PPI diário, chega à ação PETR4, ao preço, ao volume, ao tráfego e aos indicadores de atividade e emprego, e usar essa defasagem na previsão desses alvos.

Objetivos específicos:

1. Organizar a base congelada com âncora no PPI diário.
2. Medir defasagens entre PPI, preço ANP/Petrobras e PETR4.
3. Testar se choques de diesel antecipam tráfego pesado, indústria e desemprego (ODS 8).
4. Testar se choques de gasolina e tráfego leve descrevem mobilidade urbana (ODS 11).
5. Documentar o PPI como sinal público da política de preços da Petrobras (ODS 16).
6. Entregar um notebook reproduzível com modelo estatístico de série temporal.

A hipótese de trabalho é que o choque de paridade e de preço passa primeiro pela ação e, depois, pela indústria, pelo varejo, pelo emprego e pelo tráfego.

## Justificativa

Um dos integrantes do grupo criou o pipeline ETL que captura e armazena esses dados. O pipeline está ativo. Nenhum estudo foi produzido a partir deste pipeline.

A contribuição analítica é um recorte reproduzível de fontes originais, com frequência mista e âncora diária, sem base pré-processada de terceiro. A contribuição prática é a medida das defasagens para quem lê política de preços, frete e mobilidade.

O ODS 8 liga diesel, emprego e crescimento. O ODS 11 liga gasolina, tráfego leve e cidades. O ODS 16 liga a transparência da política de preços à Petrobras como instituição.

O produto do curso é um experimento executável. Não é um painel de produção.

## Descrição da base de dados

A base do curso é o arquivo `data/gas_flare.sqlite`. O arquivo é um snapshot congelado. O projeto não acessa Postgres, API ao vivo nem credencial. As séries vêm de fontes originais (governo e entidades setoriais), reunidas em 14 tabelas. A âncora temporal é o PPI diário: 5 de janeiro de 2022 a 21 de setembro de 2026, com 1.151 datas. As demais famílias entram nesse calendário de dias úteis, cada uma com a defasagem da fonte.

Quadro 1 — Famílias da base congelada

| Família | Frequência | Conteúdo | Recorte no snapshot | Fonte |
|---|---|---|---|---|
| PPI | diária | diferencial Petrobras e independentes vs. paridade, gasolina e diesel, R\$/L e % | 5 jan. 2022 a 21 set. 2026 (1.151 datas) | ABICOM |
| Ação PETR4 | diária (pregão) | abertura, máxima, mínima, fechamento, volume | 1º fev. 2022 a 18 set. 2026 (1.156 pregões) | B3 |
| Preço ANP | semanal | média ao produtor/importador, linha Brasil: gasolina A comum; diesel S-10 e S-500, R\$/L | 3 jan. 2022 a 13 set. 2026 (245 semanas por produto) | ANP |
| Preço Petrobras | semanal irregular | preço de lista às distribuidoras, sem tributos, gasolina A e diesel S-10 e S-500, R\$/L | jan. 2022 a set. 2026 (64 semanas de gasolina; 72 de diesel) | Petrobras |
| Vendas ANP | mensal | volume nacional, m³ | jan. 2022 a jul. 2026 | ANP |
| Vendas e finanças Petrobras | trimestral | volume (mbpd), receita e margem RT&M | vendas 2022T1–2026T2; finanças de 2006T1 a 2026T2 | Petrobras |
| Tráfego | mensal | índice ABCR leve, pesado e total | fev. 2022 a ago. 2026 | ABCR |
| Atividade e emprego | mensal | desemprego (PNAD), varejo (PMC), indústria (PIM-PF) | jan. 2022 a jul. 2026 | IBGE |
| Inflação | mensal | IPCA, IPCA-15 e IGP-M | jan. 2022 a ago. 2026 | IBGE / FGV via BCB |

A coleta operacional de origem é semanal. O curso usa o estado congelado. Semanas sem PDF novo da Petrobras não avançam a série de preço à distribuidora: isso é ausência de reajuste, não falha de carga. PETR4 começa em fevereiro de 2022; os NaNs de janeiro no índice do PPI são esperados. Os preços da B3 não têm ajuste por proventos. O dicionário das tabelas está em `docs/metadados_base.md`.

### Atualização da base

A revisão da Etapa 2 conferiu a base congelada contra as fontes e encontrou erros de origem. O grupo levou esses erros ao pipeline que alimenta a base, e o pipeline corrigiu os pontos abaixo:

- PPI: leituras de OCR erradas, como −13,0 R\$/L em 26 de janeiro de 2023 e percentuais de três dígitos sem o primeiro algarismo em setembro de 2026.
- Preço ANP: o diesel misturava biodiesel B-100 e diesel marítimo ao diesel rodoviário, e a gasolina incluía a gasolina Premium. A carga também somava 22 cópias da linha Brasil como se fossem regiões.
- Preço Petrobras: algumas semanas vinham de tabelas de outro produto.
- Finanças Petrobras: os trimestres 1T26 e 2T26 estavam vazios.

Depois das correções, o grupo exportou uma nova base, com corte em 21 de setembro de 2026. O Quadro 1 descreve essa base. A versão anterior fica em `data/gas_flare_old.sqlite`, e a tabela `export_changelog` registra cada correção com a causa e a maior alteração de valor.


# **Referencial Teórico**

## Conceitos

Uma série temporal é um conjunto de observações coletadas em intervalo regular. Neste projeto, o PPI e o pregão de PETR4 seguem os dias úteis, enquanto preço, volume, tráfego e atividade chegam por semana, mês ou trimestre; reunidas no mesmo calendário, essas séries formam uma série multivariada.

A leitura de cada série separa quatro componentes: a tendência, que indica se a série sobe, fica estável ou cai; a sazonalidade, que se repete em período fixo, aqui o ano; o ciclo, que se repete sem período fixo; e o ruído, o movimento irregular que sobra depois dos outros três. Uma série estacionária mantém média e variância estáveis no tempo, o que não acontece quando há tendência, sazonalidade ou ciclo. A diferença entre valores consecutivos, ou a diferença sazonal, devolve a série a essa condição, e a função de autocorrelação (ACF) e a autocorrelação parcial (PACF) mostram quanto cada valor depende dos anteriores, o que orienta a ordem do modelo.

O preço de uma ação se comporta como série caótica, em que o movimento diário é quase todo ruído e não tem período. No recorte, o retorno diário de PETR4 tem média de 0,056% e desvio padrão de 2,1%, e por isso o grupo mantém curto o horizonte do vetor diário, de 5 e 21 dias úteis.

Dois tratamentos preparam as séries para os modelos. Um buraco no calendário recebe o valor anterior, o seguinte ou a média de uma janela curta, sem que o valor medido seja removido. Além disso, a criação de variáveis (*feature engineering*) transforma as séries brutas em indicadores: o projeto pondera o desvio do PPI pelo volume de combustível vendido e soma esse indicador no tempo para formar um estoque. Os modelos recebem esses indicadores prontos, como variáveis exógenas, e o cálculo está no Diagrama de Solução.

## Modelos

O ARIMA combina uma parte autorregressiva (AR), que usa valores passados, com uma média móvel (MA), que usa erros passados, e aplica a integração (I), isto é, a diferença, até a série ficar estacionária. O SARIMA acrescenta termos sazonais quando a série repete um padrão no ano, e o SARIMAX aceita ainda variáveis exógenas, como o PPI.

O Prophet segue outro caminho e trata a série como soma de uma tendência por partes, de sazonalidades em séries de Fourier, de feriados e de regressores externos (TAYLOR; LETHAM, 2018). Como o ajuste é uma regressão no tempo, o modelo não exige série estacionária, aceita datas faltantes e recebe o PPI como regressor. Ele funciona bem em séries com tendência e ciclo anual, mas perde força em série caótica, como o preço diário de uma ação.

A rede neural recorrente (RNN) devolve a saída do passo anterior à entrada e, assim, guarda memória da sequência. A LSTM (memória de longo e curto prazo) é a forma mais usada dessa rede, mas precisa de mais dados que a família ARIMA.

O TimesFM é um modelo de fundação para séries temporais, desenvolvido pelo Google Research: um transformador do tipo decodificador que lê a série em trechos (*patches*) e prevê os trechos seguintes (DAS et al., 2024). A versão 3.0, de agosto de 2026, tem 330 milhões de parâmetros, foi pré-treinada com mais de um trilhão de pontos de séries temporais e aceita entrada multivariada com covariáveis. Ao contrário do SARIMAX, ajustado a cada série, e da LSTM, treinada nesta base, o TimesFM prevê sem treino na base do projeto (*zero-shot*). Com 1.151 datas diárias, a base é pequena para a LSTM, e o modelo pré-treinado compensa parte dessa falta. O código do TimesFM tem licença Apache 2.0, e os pesos da versão 3.0 têm licença não comercial, que permite o uso em estudo acadêmico.

## Trabalhos correlatos

Lopes et al. (2021) estudaram os fatores que moveram a ação PETR4 entre 2009 e 2020, com correlação de Pearson e regressão por mínimos quadrados ordinários. O preço do Brent, o Ibovespa e o volume de produção foram significativos, enquanto a política de preços de combustíveis apareceu só na discussão qualitativa. O trabalho explica o nível da ação, mas não faz previsão nem usa o PPI.

Puentes Márquez et al. (2021) previram a demanda de etanol combustível no Brasil e compararam o ARIMA com redes LSTM de uma e de várias camadas, em versão univariada e multivariada. A LSTM superou o ARIMA mesmo com 180 observações de treino, resultado que apoia o uso dessa rede no vetor diário, com cerca de 1.100 observações.

Serrano et al. (2025) previram o consumo de gasolina e de etanol no Brasil com séries de 72, 144 e 263 meses e compararam ARIMA/SARIMA, Holt-Winters, ETS, TBATS, Prophet, Orbit, N-BEATS e TFT. O TBATS teve o menor erro para a gasolina (MAPE de 3,22%) e o Orbit para o etanol (MAPE de 6,77%); como nenhum modelo venceu nos dois combustíveis, o estudo reforça a comparação de mais de uma técnica.

Os três trabalhos usam uma frequência só, não usam o PPI diário e não testam um modelo de fundação. O projeto se diferencia ao juntar dia, semana, mês e trimestre com âncora no PPI e ao comparar SARIMAX, Prophet, LSTM e TimesFM.

# **Diagrama de Solução**

O pipeline segue as cinco fases da análise de séries temporais: coleta, pré-processamento, análise exploratória, modelagem e avaliação. O produto desvio × volume diário e o estoque são calculados antes dos modelos, que recebem esses indicadores prontos.

```
1. COLETA
   base congelada data/gas_flare.sqlite
   âncora: PPI diário (ABICOM), 1.151 datas, corte em 21/09/2026
        |
        v
2. PRÉ-PROCESSAMENTO
   alinhamento ao calendário de dias úteis
   buraco de calendário -> valor vizinho
   conferência da leitura OCR do PPI
   volume diário (ANP: m³ do mês; Petrobras: mil barris por dia)
   produto do dia = desvio em R$/L × volume diário
   soma dos produtos = estoque
   normalização de escala (só para a LSTM)
        |
        v
3. ANÁLISE EXPLORATÓRIA E VISUALIZAÇÃO
   tendência, sazonalidade, ciclo e ruído
   estacionariedade e diferença
   ACF e PACF
   correlação cruzada PPI × alvo -> defasagem
   gráfico de linha, de caixa e mapa de calor
        |
        v
4. MODELAGEM (um corte temporal por vetor)
        +-------------+-------------+-------------+
        |             |             |             |
        v             v             v             v
     mercado        preço       atividade     companhia
     dia útil       semana         mês        trimestre
     SARIMAX        SARIMAX      SARIMAX       SARIMAX
     LSTM           Prophet      Prophet
     TimesFM
        |             |             |             |
        +-------------+------+------+-------------+
                             |
                             v
5. AVALIAÇÃO E VALIDAÇÃO
   erro absoluto médio, erro quadrático médio e sua raiz
   gráfico da série real contra a prevista
   horizonte próprio de cada vetor
```

## Pré-processamento

**PPI.** O cartão da ABICOM traz o desvio em R\$/L e em %, e o percentual é calculado sobre o preço vigente da Petrobras: (preço Petrobras − paridade) / preço Petrobras. A razão entre as duas colunas fica constante entre reajustes, o que confirma essa conta. Em 5 de maio de 2026, por exemplo, a gasolina marca −R\$ 2,22/L e −88%, com o preço Petrobras perto de R\$ 2,52/L e a paridade perto de R\$ 4,74/L; o reajuste até a paridade seria de 88% sobre o preço vigente. Em 2026, o preço Petrobras usado pelo PPI é o preço com desconto das tabelas da Petrobras, enquanto essa coluna existe, ao passo que a série de preço às distribuidoras usa o preço de lista, e por isso as duas séries diferem nesse ano.

**Leitura OCR.** Os valores do PPI saem da imagem da ABICOM por OCR. A revisão da Etapa 2 encontrou leituras erradas, que o pipeline de origem corrigiu contra a imagem, e uma regra passou a comparar cada percentual com o preço de referência dos dez dias anteriores. Quando a divergência passa de 1,5 ponto percentual, a linha vai para conferência manual, marcada na coluna `man_insp`.

**Volume diário.** O desvio em R\$/L entra multiplicado por um volume diário, porque a soma do desvio sem volume não mede o combustível que circulou. Como a base não publica venda diária, os vetores de mercado, preço e atividade tiram o volume diário da venda mensal da ANP, com o volume em m³ vezes 1.000, dividido pelos dias do mês. A ANP soma todos os fornecedores, e a Petrobras responde por cerca de 58% desse volume no 3º trimestre de 2025; por isso o vetor da companhia usa a taxa da própria Petrobras, em mil barris por dia, vezes 159 L por barril. O produto do dia é o desvio vezes esse volume, em R\$, e a série acumulada soma esses produtos. O percentual fica fora da multiplicação, porque a divisão pelo preço já ocorreu, e, na previsão, a última taxa diária conhecida se repete.

**Escala e calendário.** A LSTM recebe as séries normalizadas entre 0 e 1, enquanto o SARIMAX, o Prophet e o TimesFM recebem os valores originais. Como o pregão de PETR4 começa em 1º de fevereiro de 2022, as datas de janeiro de 2022 saem do treino do vetor diário.

## Análise exploratória

A leitura de cada vetor separa tendência, sazonalidade, ciclo e ruído, testa a estacionariedade e mede a ACF e a PACF. A EDA preliminar já mostra três padrões. A venda de diesel da ANP tem pico no 3º trimestre em 2022 e em 2023, e a participação da Petrobras no volume da ANP cai cerca de 0,9 ponto percentual por trimestre desde 2022. Em 2026, a variação semanal típica do preço ANP é três a quatro vezes maior que em 2022–2025; como esse preço é média de produtores e importadores e o preço Petrobras fica muito abaixo da paridade nesse ano, uma causa provável é a parcela importada, que muda de semana para semana. Os demais componentes ficam para a Etapa 3.

A correlação cruzada entre o PPI e cada alvo mede em quantos dias, semanas ou meses o alvo responde ao choque, e a defasagem de maior correlação entra no modelo como PPI defasado.

## Modelagem

Como a base junta dia, semana, mês e trimestre, a proposta separa quatro vetores, para não levar o horizonte de 5 e 21 dias úteis às séries mais lentas.

| Vetor | Alvo | Entrada | Horizonte | Modelo |
|---|---|---|---|---|
| Mercado | fechamento de PETR4 | PPI defasado e desvio × volume diário | 5 e 21 dias úteis | SARIMAX, LSTM e TimesFM 3.0 |
| Preço | preço da ANP e preço da Petrobras | PPI da semana, defasado | 4 e 8 semanas | SARIMAX sazonal e Prophet |
| Atividade | tráfego pesado, indústria, varejo e desemprego | desvio × volume diário, somado no mês | 1 e 3 meses | SARIMAX sazonal e Prophet |
| Companhia | volume, receita RT&M, lucro bruto e margem | desvio × volume diário da Petrobras, somado no trimestre | 2º trimestre de 2026 | SARIMAX, conferido com o resultado publicado |

O SARIMAX recebe o PPI defasado e o produto desvio × volume como variáveis exógenas. Quando a defasagem é igual ou maior que o horizonte, o teste usa só valores de PPI conhecidos na data da previsão; quando é menor, o último PPI conhecido se repete. A ordem do modelo sai de uma busca automática (`auto_arima`), e o termo sazonal entra quando a EDA mostra repetição anual, caso esperado em preço, volume e tráfego. Nos vetores de preço e atividade, o Prophet usa o mesmo corte e o mesmo horizonte do SARIMAX, com sazonalidade anual e com o PPI defasado e o produto desvio × volume como regressores externos (`add_regressor`).

A LSTM é treinada só no vetor diário, com busca em grade (*grid search*) de taxa de aprendizado, número de épocas e tamanho da janela. O TimesFM 3.0 roda duas vezes nesse vetor, sem treino: com PETR4 sozinha e com PETR4 acompanhada do PPI da gasolina e do PPI do diesel como covariáveis. A história desde fevereiro de 2022 entra no contexto, para que o ciclo anual esteja presente. Os pesos são baixados na execução e ficam fora do repositório; se a instalação impedir essa rodada, o vetor diário fica com o SARIMAX e a LSTM. As séries mensal, com cerca de 55 pontos, e trimestral, com 17, são curtas para a LSTM e para o TimesFM, e por isso a mensal fica com o SARIMAX e o Prophet, e a trimestral só com o SARIMAX.

## Avaliação e partição

Em todos os vetores, a avaliação usa o erro absoluto médio, o erro quadrático médio, a raiz desse erro e o gráfico da série real contra a prevista.

A partição usa só a base congelada `data/gas_flare.sqlite`, com corte em 21 de setembro de 2026, sem nenhuma observação nova. O corte é temporal e há um por vetor; no vetor diário, ele é o mesmo para o SARIMAX, a LSTM e o TimesFM 3.0. O teste tem o tamanho do horizonte mais longo, e o horizonte curto é medido dentro desse mesmo teste: 5 dias úteis dentro dos 21, 4 semanas dentro das 8 e 1 mês dentro dos 3.

Alguns pontos ficam de fora para manter as séries alinhadas. O PPI de 21 de setembro de 2026 não entra, porque o pregão de PETR4 termina em 18 de setembro, e o tráfego de agosto de 2026 também não, porque as outras séries mensais param em julho. No resultado da Petrobras, o teste é o 2º trimestre de 2026, último trimestre publicado, cuja receita inclui R\$ 9,7 bilhões de subvenção ao diesel e R\$ 0,8 bilhão à gasolina, fato sem paralelo no treino. Como esse teste tem um ponto só, o erro desse vetor serve apenas como indicação.

| Vetor | Treino | Teste |
|---|---|---|
| Mercado | 1º fev. 2022 a 19 ago. 2026 | 20 ago. 2026 a 18 set. 2026 (21 dias úteis) |
| Preço | até a semana encerrada em 19 jul. 2026 | 20 jul. 2026 a 13 set. 2026 (8 semanas) |
| Atividade | até abr. 2026 | mai., jun. e jul. 2026 |
| Companhia | 1º trim. 2022 ao 1º trim. 2026 | 2º trim. 2026, resultado já publicado |

Os números e os ajustes entram na Etapa 3.

# **Cronograma**

Os trabalhos do grupo serão desenvolvidos segundo o seguinte cronograma:

| Tarefa | Etapa | Início | Término | Dias |
|---|---|---|---|---|
| Redigir referencial, pipeline e cronograma | 2 | 22/09/2026 | 27/09/2026 | 6 |
| Montar o volume diário, o produto desvio × volume e a leitura dos componentes, com ACF, PACF e correlação cruzada | 3 | 29/09/2026 | 05/10/2026 | 7 |
| Redigir a EDA e o pré-processamento no notebook | 3 | 29/09/2026 | 05/10/2026 | 7 |
| Ajustar o SARIMAX diário e o semanal e treinar a LSTM diária, com busca de hiperparâmetros | 3 | 06/10/2026 | 12/10/2026 | 7 |
| Rodar o TimesFM 3.0 no vetor diário, com a história desde 2022 | 3 | 13/10/2026 | 16/10/2026 | 4 |
| Ajustar o SARIMAX mensal, o Prophet semanal e o mensal, e prever o 2º trimestre de 2026 da Petrobras | 3 | 17/10/2026 | 26/10/2026 | 10 |
| Atualizar o pipeline e o cronograma | 3 | 20/10/2026 | 26/10/2026 | 7 |
| Comparar os três modelos no vetor diário e o SARIMAX com o Prophet nos vetores de preço e atividade | 4 | 27/10/2026 | 02/11/2026 | 7 |
| Analisar a defasagem e o modelo de melhor resultado | 4 | 03/11/2026 | 09/11/2026 | 7 |
| Redigir resultados, discussão e conclusão | 4 | 10/11/2026 | 16/11/2026 | 7 |
| Preparar slides e roteiro do vídeo | 4 | 17/11/2026 | 23/11/2026 | 7 |
| Gravar e publicar o vídeo e revisar o notebook | 4 | 24/11/2026 | 30/11/2026 | 7 |

*Os dias 28/09, 26/10 e 30/11 são marcos do curso.*

# **EDA e Pré-processamento dos dados**

*(Etapa 3.)* Exploração e análise dos dados. Qualidade, limites e recortes. Transformações, junções e encodes.


In [ ]:
# Códigos aqui (etapa 3)


# **Modelos**

*(Etapas 3 e 4.)* Descrição metodológica da criação, do refinamento e da validação dos modelos.


In [ ]:
# Códigos aqui (etapas 3 e 4)


# **Resultados**

*(Etapa 4.)* Métricas, tabelas comparativas e leitura dos resultados.


# **Discussão e Conclusão**

*(Etapa 4.)* Qualidades e limites do trabalho. Alcance do objetivo. Melhorias possíveis.


# **Apresentação**

*(Etapa 4.)* Link do vídeo (apresentação com slides).


# **Referências**

ASSOCIAÇÃO BRASILEIRA DE CONCESSIONÁRIAS DE RODOVIAS. *Índice ABCR*. São Paulo: ABCR, 2026. Disponível em: https://melhoresrodovias.org.br/. Acesso em: 19 ago. 2026.

ASSOCIAÇÃO BRASILEIRA DOS IMPORTADORES DE COMBUSTÍVEIS. *PPI*. Rio de Janeiro: ABICOM, 2026. Disponível em: https://abicom.com.br/categoria/ppi/. Acesso em: 19 ago. 2026.

AGÊNCIA NACIONAL DO PETRÓLEO, GÁS NATURAL E BIOCOMBUSTÍVEIS. *Preços de produtores e importadores de derivados de petróleo e biodiesel*. Brasília: ANP, 2026. Disponível em: https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/precos-de-produtores-e-importadores-de-derivados-de-petroleo-e-biodiesel. Acesso em: 27 set. 2026.

AGÊNCIA NACIONAL DO PETRÓLEO, GÁS NATURAL E BIOCOMBUSTÍVEIS. *Vendas de combustíveis (m³)*. Brasília: ANP, 2026. Disponível em: https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-estatisticos. Acesso em: 19 ago. 2026.

BANCO CENTRAL DO BRASIL. *Sistema Gerenciador de Séries Temporais (SGS)*. Brasília: BCB, 2026. Disponível em: https://www3.bcb.gov.br/sgspub/. Acesso em: 19 ago. 2026.

B3 S.A. – BRASIL, BOLSA, BALCÃO. *Cotações históricas*. São Paulo: B3, 2026. Disponível em: https://www.b3.com.br/pt_br/market-data-e-indices/servicos-de-dados/market-data/historico/mercado-a-vista/cotacoes-historicas/. Acesso em: 19 ago. 2026.

DAS, A.; KONG, W.; SEN, R.; ZHOU, Y. A decoder-only foundation model for time-series forecasting. In: INTERNATIONAL CONFERENCE ON MACHINE LEARNING, 41., 2024, Viena. *Proceedings* [...]. [S. l.]: PMLR, 2024. v. 235. Disponível em: https://arxiv.org/abs/2310.10688. Acesso em: 25 set. 2026.

GOOGLE RESEARCH. *TimesFM*: time series foundation model. [S. l.]: GitHub, 2026. Disponível em: https://github.com/google-research/timesfm. Acesso em: 25 set. 2026.

HYNDMAN, R. J.; ATHANASOPOULOS, G. *Forecasting*: principles and practice. 3. ed. Melbourne: OTexts, 2021. Disponível em: https://otexts.com/fpp3/. Acesso em: 24 ago. 2026.

INSTITUTO BRASILEIRO DE GEOGRAFIA E ESTATÍSTICA. *SIDRA*: Sistema IBGE de Recuperação Automática. Rio de Janeiro: IBGE, 2026. Disponível em: https://sidra.ibge.gov.br/. Acesso em: 19 ago. 2026.

LOPES, A. O.; VIEGAS, T. O. C.; CONTE FILHO, C. G.; CARVALHO, V. S. Fatores influenciando preços de ações Petrobras (PETR4) entre 2009 e 2020. *Research, Society and Development*, v. 10, n. 7, 2021. Disponível em: https://rsdjournal.org/index.php/rsd/article/view/16294. Acesso em: 25 set. 2026.

ORGANIZAÇÃO DAS NAÇÕES UNIDAS. *Sustainable Development Goals*. Nova York: ONU, 2024. Disponível em: https://www.undp.org/sustainable-development-goals. Acesso em: 24 ago. 2026.

PETROBRAS. *Central de resultados*. Rio de Janeiro: Petrobras, 2026. Disponível em: https://www.investidorpetrobras.com.br/resultados-e-comunicados/central-de-resultados/. Acesso em: 19 ago. 2026.

PETROBRAS. *Tabelas de preços de venda às distribuidoras*. Rio de Janeiro: Petrobras, 2026. Disponível em: https://precos.petrobras.com.br. Acesso em: 19 ago. 2026.

PUENTES MÁRQUEZ, J.; OLIVEIRA RIBEIRO, C. de; RUELAS SANTOYO, E.; FIGUEROA FERNÁNDEZ, V. Ethanol fuel demand forecasting in Brazil using a LSTM recurrent neural network approach. *IEEE Latin America Transactions*, v. 19, n. 4, p. 551-558, 2021. Disponível em: https://latamt.ieeer9.org/index.php/transactions/article/view/3713. Acesso em: 25 set. 2026.

SERRANO, A. L. M. et al. Forecasting ethanol and gasoline consumption in Brazil: advanced temporal models for sustainable energy management. *Energies*, v. 18, n. 6, 1501, 2025. DOI: https://doi.org/10.3390/en18061501.

SILVA, L. A.; PERES, S. M.; BOSCARIOLI, C. *Introdução à mineração de dados com aplicações em R*. 1. ed. São Paulo: LTC, 2016.

TAYLOR, S. J.; LETHAM, B. Forecasting at scale. *The American Statistician*, v. 72, n. 1, p. 37-45, 2018. DOI: https://doi.org/10.1080/00031305.2017.1380080.